# CS570 Week 4 Lab: DataFrames, Spark SQL & Data Investigation

**Points: 100** | **Due: See Canvas**

---

### Instructions

1. Run every code cell and keep all outputs visible
2. Write your code in cells marked `# YOUR CODE`
3. Fill in the **Results Sheet** at the bottom with your exact values
4. Export as PDF (File → Print Preview → Save as PDF)
5. Submit the PDF to Canvas

**Grading:** The Results Sheet is your scorecard. Code cells are your evidence.

---

## Part 0: Setup

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Task:** Create a SparkSession. You did this in Week 3.

In [ ]:
# YOUR CODE: Create a SparkSession called 'spark'


In [ ]:
# Dataset: spotify.csv (same as Week 3)
# If you don't have it, download from the Week 3 lab instructions
data_path = "spotify.csv"

---
## Part 1: Loading & Schema Validation (15 pts)

In production, you always need to validate that your data loaded correctly.

### 1.1 Load with inferSchema

In [ ]:
start = time.time()

df_infer = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(data_path)

df_infer.count()  # force evaluation

infer_time = time.time() - start
print(f"inferSchema load time: {infer_time:.2f} seconds")

### 1.2 Examine the schema

**Task:** Print the schema and show the first few rows. Report the number of columns and rows.

In [ ]:
# YOUR CODE: Print the schema that Spark inferred


In [ ]:
# YOUR CODE: Show the first 5 rows


In [ ]:
# YOUR CODE: How many columns? How many rows?


**Task:** Look at the columns. Are there any that don't add analytical value? Drop them.

In [ ]:
# YOUR CODE: Drop any columns that don't add value
# Reassign to df_infer


### 1.3 Load with explicit schema

Explicit schemas are faster and catch data problems early.

In [ ]:
explicit_schema = StructType([
    StructField("_c0", IntegerType(), True),
    StructField("track_id", StringType(), True),
    StructField("artists", StringType(), True),
    StructField("album_name", StringType(), True),
    StructField("track_name", StringType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("duration_ms", IntegerType(), True),
    StructField("explicit", BooleanType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy", DoubleType(), True),
    StructField("key", IntegerType(), True),
    StructField("loudness", DoubleType(), True),
    StructField("mode", IntegerType(), True),
    StructField("speechiness", DoubleType(), True),
    StructField("acousticness", DoubleType(), True),
    StructField("instrumentalness", DoubleType(), True),
    StructField("liveness", DoubleType(), True),
    StructField("valence", DoubleType(), True),
    StructField("tempo", DoubleType(), True),
    StructField("time_signature", IntegerType(), True),
    StructField("track_genre", StringType(), True)
])

start = time.time()

df = spark.read \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .schema(explicit_schema) \
    .csv(data_path)

df.count()  # force evaluation

explicit_time = time.time() - start
print(f"Explicit schema load time: {explicit_time:.2f} seconds")
print(f"Speedup: {infer_time / explicit_time:.2f}x")

**Task:** Verify row counts match, then drop the same useless column from df.

In [ ]:
# YOUR CODE: Verify both loaded the same number of rows
# Then drop the useless column from df as well


**→ Results Sheet: R1, R2, R3, R4**

---
## Part 2: Data Integrity Investigation (30 pts)

How many unique records does this dataset actually contain?

### 2.1 Find the null row

**Task:** Find which columns have null values. Display the row(s) with nulls. Remove them.

In [ ]:
# YOUR CODE: Count nulls in each column


In [ ]:
# YOUR CODE: Display the row(s) that have null values


In [ ]:
# YOUR CODE: Remove the null row(s), store as df_clean
# Print row count before and after

df_clean = # your code

print(f"Rows before: {df.count()}")
print(f"Rows after: {df_clean.count()}")

**→ Results Sheet: R5, R6**

### 2.2 Examine the artists column

**Task:** Look closely at the `artists` column. What do you notice about tracks with multiple artists?

In [ ]:
# YOUR CODE: Show some rows where it looks like multiple artists are involved
# Hint: look for a pattern in how they're stored


**→ Results Sheet: R7 — How are multiple artists stored in the artists column?**

### 2.3 How many unique tracks?

**Task:** Determine how many unique tracks are in the dataset. But first — what does "unique" mean? You decide.

In [ ]:
# YOUR CODE: Count unique tracks
# First, write your definition:
# MY DEFINITION: A unique track is defined by _______________
#
# Now implement it:


In [ ]:
# YOUR CODE: Compare total rows vs unique tracks


In [ ]:
# YOUR CODE: Show the distribution — how many tracks appear 1x, 2x, 3x, etc.


**→ Results Sheet: R8, R9, R10**

### 2.4 Investigate the duplicates

**Task:** Pick a track that appears multiple times. Show all its rows. Determine: do the audio features (energy, danceability) differ? Does the genre differ?

In [ ]:
# YOUR CODE: Pick a track that appears 4+ times, show all its rows
# Include: track_id, track_name, artists, track_genre, energy, danceability


**Task:** Prove whether duplicates have identical audio features or different ones.

In [ ]:
# YOUR CODE: How many track_ids have DIFFERENT energy values across duplicates?
# Hint: groupBy track_id, use countDistinct on energy, filter where > 1


In [ ]:
# YOUR CODE: How many track_ids have DIFFERENT genre values across duplicates?


**→ Results Sheet: R11, R12, R13**

---
## Part 3: DataFrame Operations & SQL (20 pts)

In [ ]:
# Register for SQL
df_clean.createOrReplaceTempView("tracks")

### 3.1 Impact of duplicates on aggregations

**Task:** Create a deduplicated DataFrame based on your definition of unique track. Then calculate average popularity both ways and compare.

In [ ]:
# YOUR CODE: Deduplicate based on YOUR definition of unique track
# Hint: dropDuplicates([...]) is easiest

df_dedup = # your code

print(f"Rows before dedup: {df_clean.count()}")
print(f"Rows after dedup: {df_dedup.count()}")

In [ ]:
# YOUR CODE: Calculate average popularity WITH duplicates and WITHOUT
# Report both values rounded to 2 decimals


**→ Results Sheet: R14, R15**

### 3.2 Normalize popularity

**Task:** All audio features are 0.0–1.0. But popularity is 0–100. Add a new column `popularity_norm` that scales it to 0.0–1.0.

In [ ]:
# YOUR CODE: Add popularity_norm column using withColumn
# Show 5 rows with track_name, popularity, popularity_norm to verify


### 3.3 Unique tracks per genre

**Task:** Using Spark SQL, find how many unique tracks each genre has. Show top 5.

In [ ]:
# YOUR CODE: SQL query for unique track count per genre
# Remember: count(*) ≠ unique tracks in denormalized data

spark.sql("""
    -- YOUR SQL HERE
""").show(5, truncate=False)

**→ Results Sheet: R16**

---
## Part 4: Catalyst Optimizer (15 pts)

### 4.1 Optimize this query

**Task:** Look at this inefficient query. Write an optimized version.

In [ ]:
# Inefficient query:
query_a = df_clean \
    .filter(col("popularity") > 50) \
    .filter(col("energy") > 0.5) \
    .filter(col("popularity") > 70) \
    .select("track_name", "popularity", "energy")

In [ ]:
# YOUR CODE: Write an optimized version
# Think: redundant filters, order of operations

query_b = # your optimized version

### 4.2 Compare execution plans

**Task:** Run explain() on both queries. What do you notice?

In [ ]:
# YOUR CODE: Run explain() on both

print("=== Query A (original): ===")
query_a.explain()

print("\n=== Query B (your version): ===")
query_b.explain()

In [ ]:
# YOUR CODE: Verify both return the same number of rows


**→ Results Sheet: R17, R18**

---
## Part 5: Pipeline Challenge (20 pts)

**Task:** Build a pipeline to answer this question:

> Which individual artists appear in at least 5 different genres AND have an average track popularity above 50?

**Important:** Remember what you discovered about the `artists` column in Part 2. A track like "Artist A;Artist B;Artist C" should count as appearances for A, B, and C separately.

Requirements:
- Start from df_clean
- Parse the artists column properly
- Show: artist, genre_count, avg_popularity (rounded to 2 decimals)
- Order by genre_count descending, then by avg_popularity descending
- Show top 10

*Hint: Look up `split()` and `explode()` functions.*

In [ ]:
# YOUR CODE: Build your pipeline


In [ ]:
# YOUR CODE: Show the result


**→ Results Sheet: R19, R20**

---
## Part 6: Cleanup

In [ ]:
spark.stop()
print("Spark stopped")

---
---

# RESULTS SHEET

**Fill in every value below. This is what gets graded.**

| # | Question | Your Answer | Points |
|---|----------|-------------|--------|
| R1 | inferSchema load time (seconds) | | 3 |
| R2 | Explicit schema load time (seconds) | | 3 |
| R3 | Speedup ratio (R1 / R2) | | 4 |
| R4 | Number of columns (after dropping useless one) | | 5 |
| R5 | Which columns have nulls? | | 3 |
| R6 | Row count after removing nulls | | 3 |
| R7 | How are multiple artists stored? (describe the format) | | 4 |
| R8 | Your unique track count | | 4 |
| R9 | Your definition of "unique track" (one sentence) | | 5 |
| R10 | Max times a single track appears | | 4 |
| R11 | Track_ids with differing energy across duplicates | | 4 |
| R12 | Track_ids with differing genre across duplicates | | 4 |
| R13 | Why do duplicates exist? (one sentence) | | 5 |
| R14 | Avg popularity WITH duplicates (rounded to 2) | | 5 |
| R15 | Avg popularity WITHOUT duplicates (rounded to 2) | | 5 |
| R16 | Genre with most unique tracks + that count | | 6 |
| R17 | Filters in Query A's physical plan | | 6 |
| R18 | Are the plans identical? (yes/no + why) | | 7 |
| R19 | Artist appearing in most genres (meeting criteria) | | 10 |
| R20 | That artist's avg popularity (rounded to 2 decimals) | | 10 |
| | | **TOTAL** | **100** |